In [ ]:
# Pull latest project code from GitHub
import os
import shutil
import subprocess

REPO_URL = "https://github.com/nayanjha16/CodeGen-Implementations-May_26.git"
BRANCH = "Group-44"  # canonical capstone branch
WORK_PROJECT = "/kaggle/working/project"


def run(cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)


if os.path.isdir(os.path.join(WORK_PROJECT, ".git")):
    run(["git", "-C", WORK_PROJECT, "fetch", "origin", BRANCH])
    run(["git", "-C", WORK_PROJECT, "checkout", BRANCH])
    run(["git", "-C", WORK_PROJECT, "pull", "--ff-only", "origin", BRANCH])
else:
    if os.path.exists(WORK_PROJECT):
        shutil.rmtree(WORK_PROJECT)
    run([
        "git", "clone",
        "--branch", BRANCH,
        "--single-branch",
        "--depth", "1",
        REPO_URL,
        WORK_PROJECT,
    ])

os.chdir(WORK_PROJECT)
print("Project root:", os.getcwd())
print("Top-level files:", os.listdir("."))
print("GitHub pull complete.")

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # force single GPU

In [ ]:
import os

WORK_PROJECT = "/kaggle/working/project"
os.chdir(WORK_PROJECT)

required = ["src", "scripts", "configs", "requirements.txt"]
missing = [x for x in required if not os.path.exists(x)]
if missing:
    raise FileNotFoundError(f"Missing required items: {missing}. Run the GitHub pull cell first.")

# Reduce batch size for Kaggle GPU memory
cfg_path = os.path.join(WORK_PROJECT, "configs", "default.yaml")
with open(cfg_path, "r", encoding="utf-8") as f:
    cfg = f.read()
cfg = cfg.replace("per_device_train_batch_size: 8", "per_device_train_batch_size: 2")
cfg = cfg.replace("per_device_eval_batch_size: 8", "per_device_eval_batch_size: 2")
cfg = cfg.replace("gradient_accumulation_steps: 4", "gradient_accumulation_steps: 8")
with open(cfg_path, "w", encoding="utf-8") as f:
    f.write(cfg)

print("Working project root:", WORK_PROJECT)
print("Updated configs/default.yaml for Kaggle GPU")
print("Setup complete. Continue with next cell.")

In [ ]:
import os
import sys
from pathlib import Path

WORK_PROJECT = "/kaggle/working/project"
os.chdir(WORK_PROJECT)
os.environ["PYTHONPATH"] = WORK_PROJECT
sys.path.insert(0, WORK_PROJECT)


def _resolve_hf_token() -> str:
    for key in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        if os.environ.get(key):
            return os.environ[key]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return ""


# Fix peft/torchao conflict on Kaggle
!pip uninstall -y torchao
!pip install -q -r requirements.txt

env_lines = [
    "MODEL_NAME=Salesforce/codegen-350M-multi",
    "BERTSCORE_MODEL_NAME=distilbert-base-uncased",
    "TEND_DATASET_ID=care2achieve/tend",
    "TEND_CACHE_DIR=/kaggle/working/data/cache/tend",
    "MODELS_BASE_DIR=/kaggle/working/models/base",
    "MODELS_CHECKPOINTS_DIR=/kaggle/working/models/checkpoints",
    "RESULTS_DIR=/kaggle/working/results",
]
hf_token = _resolve_hf_token()
if hf_token:
    env_lines.append(f"HF_TOKEN={hf_token}")
    os.environ["HF_TOKEN"] = hf_token
    print("HF_TOKEN loaded (faster Hub downloads)")
else:
    print("HF_TOKEN not set (optional; add as Kaggle secret for faster downloads)")

Path(".env").write_text("\n".join(env_lines) + "\n", encoding="utf-8")
print("Created .env in", WORK_PROJECT)
print("Deps installed. Continue with next cell.")

In [ ]:
# Optional preflight checks (model cache + TEND dataset)
!python scripts/inspect_lora_modules.py
!python scripts/test_tend_loader.py

In [ ]:
import os
import subprocess

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

CHECKPOINT_RUN = "v4"
cmd = [
    "python", "scripts/train_all_lora.py",
    "--version", CHECKPOINT_RUN,
    "--device", "cuda:0",
    "--no-mlflow",
]
print("$", " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import os
import shutil
from pathlib import Path

CHECKPOINT_RUN = "v4"
WORK_PROJECT = "/kaggle/working/project"
CHECKPOINT_DIR = f"/kaggle/working/models/checkpoints/{CHECKPOINT_RUN}"
ZIP_BASE = f"/kaggle/working/{CHECKPOINT_RUN}_adapters"

os.chdir(WORK_PROJECT)

if not Path(CHECKPOINT_DIR).is_dir():
    raise FileNotFoundError(
        f"Checkpoint dir not found: {CHECKPOINT_DIR}. Run the training cell first."
    )

# Package adapters for download from Kaggle Output panel
shutil.make_archive(ZIP_BASE, "zip", CHECKPOINT_DIR)
zip_path = ZIP_BASE + ".zip"
print("Download from Kaggle Output:")
print(" ", zip_path)

required = ("adapter_config.json", "adapter_model.safetensors")
for task in ("text2sql", "sql2nosql", "nosql2doc"):
    task_dir = Path(CHECKPOINT_DIR) / task
    if not task_dir.is_dir():
        raise FileNotFoundError(f"Missing adapter dir: {task_dir}")
    missing = [name for name in required if not (task_dir / name).exists()]
    if missing:
        raise FileNotFoundError(f"{task} missing files: {missing}")
    print(f"{task}: OK ({', '.join(sorted(p.name for p in task_dir.iterdir() if p.is_file()))})")